# Activity 4 — Fixing a failed lookup

This activity shows how a booking lookup tool can fail. The assistant uses a tool (a booking lookup) to answer. Run the cells from top to bottom in a new kernel. The first run is intentionally broken; inspect its activity log before correcting it.

## Purpose
Find a failed lookup in the activity log and correct the tool.

## What to expect
The first tool asks for `booking_id`, but the assistant sends `booking_ref`, so the tool fails with an error and the answer reports it. The corrected tool asks for `booking_ref` and returns the booking. The activity-log names are `activity-4-tool-call-broken` and `activity-4-tool-call-fixed`.

## Bad practice
Guess the name of the detail the tool expects, and don't check the tool step when it fails.

## Good practice
Make sure the tool asks for the same detail the assistant sends, and check the tool step's input, status, and error.

## Why the good practice works
The activity log shows what the assistant sent and what the tool tried to read. The mismatch and the error are visible, so you can fix the tool instead of the assistant's wording.

## Deliverable
Both activity-log links, the evidence from the tool step, and the tool change.


In [1]:
import json
from IPython.display import Markdown, display
from experiment_utils import langfuse, print_summary, run_tool_loop, show_trace


In [2]:
question = "What is the status of booking SKY-1042?"
broken = run_tool_loop(question, "broken")
print("details sent to the tool:", broken["tool_args"])
print("tool result:", broken["tool_result"])
print("final answer:", broken["answer"])
show_trace(broken["trace_id"])


details sent to the tool: {'booking_ref': 'SKY-1042'}
tool result: {'error': "KeyError: 'booking_id'"}
final answer: I tried to look up booking **SKY-1042**, but the lookup failed with this error:

> `KeyError: 'booking_id'`

This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more.
activity log: http://localhost:3000/project/langfuse-observability/traces/3477fc6fa4ed5eb452562b4a38f5de2f


'http://localhost:3000/project/langfuse-observability/traces/3477fc6fa4ed5eb452562b4a38f5de2f'

## What went wrong?

Open the first activity log and look at the `get_booking` tool step. Compare the details the assistant sent with the details the tool tried to read. Correct the tool, not the assistant's wording.


In [3]:
fixed = run_tool_loop(question, "fixed")
print("details sent to the tool:", fixed["tool_args"])
print("tool result:", fixed["tool_result"])
print("final answer:", fixed["answer"])
show_trace(fixed["trace_id"])
rows = ["| run | details sent to the tool | tool result/error | final answer | activity log |",
        "|---|---|---|---|---|"]
for run in [broken, fixed]:
    args = json.dumps(run["tool_args"]).replace("|", "\\|")
    result = json.dumps(run["tool_result"]).replace("|", "\\|")
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['label']} | {args} | {result} | {answer} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))


details sent to the tool: {'booking_ref': 'SKY-1042'}
tool result: {'status': 'confirmed', 'origin': 'Lisbon', 'destination': 'Madrid', 'date': '2026-12-01', 'passengers': 2, 'cabin': 'economy'}
final answer: Booking **SKY-1042** is **confirmed**.

- **Route:** Lisbon → Madrid
- **Date:** December 1, 2026
- **Passengers:** 2
- **Cabin:** Economy

Let me know if you'd like any other details about this booking.
activity log: http://localhost:3000/project/langfuse-observability/traces/2e1b1c2f539179d693c1ea089a82c1d3


| run | details sent to the tool | tool result/error | final answer | activity log |
|---|---|---|---|---|
| before fix | {"booking_ref": "SKY-1042"} | {"error": "KeyError: 'booking_id'"} | I tried to look up booking **SKY-1042**, but the lookup failed with this error:  > `KeyError: 'booking_id'`  This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more. | http://localhost:3000/project/langfuse-observability/traces/3477fc6fa4ed5eb452562b4a38f5de2f |
| after fix | {"booking_ref": "SKY-1042"} | {"status": "confirmed", "origin": "Lisbon", "destination": "Madrid", "date": "2026-12-01", "passengers": 2, "cabin": "economy"} | Booking **SKY-1042** is **confirmed**.  - **Route:** Lisbon → Madrid - **Date:** December 1, 2026 - **Passengers:** 2 - **Cabin:** Economy  Let me know if you'd like any other details about this booking. | http://localhost:3000/project/langfuse-observability/traces/2e1b1c2f539179d693c1ea089a82c1d3 |

## Wrap-up

Run this cell to print the session and activity-log links, then send the remaining activity logs to Langfuse. Submit the links and your findings.


In [4]:
print_summary(4)
langfuse.flush()
print("activity logs sent")

session activity-4-7907c7fc
activity-4-broken: http://localhost:3000/project/langfuse-observability/traces/3477fc6fa4ed5eb452562b4a38f5de2f
activity-4-fixed: http://localhost:3000/project/langfuse-observability/traces/2e1b1c2f539179d693c1ea089a82c1d3
activity logs sent
